# Notebook 00: Download Macroeconomic Data from FRED 

**Author:** James Zhang

**Project** Kalshi CPI Prediction Engine

---

## Purpose

This notebook downloads, cleans, and aligns macroeconomic time series data from FRED to support inflation forecasting for Kalshi's CPI event markets. 

## Wha we doin

- We finna connect to FRED API using an API key
- Download a bunch of data (CPI, Fed Funds Rate, Unemployment, oil prices, and M2)
- Convert raw JSON responses into pandas Series
- Combine all da series into a single time-indexed DataFrame
- Save the raw macro dataset to 'data/macro_raw.parquet' for later

In [1]:
## IMPORTS + LOAD API KEY

import os
from datetime import datetime

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="/Users/jameszhang/dev/kalshi-cpi-engine/.env", override=True)

FRED_API_KEY = os.getenv("FRED_API_KEY")

FRED_API_KEY


'ce9938779e98ca662ea32c7e7959fe3c'

In [2]:
BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

def fetch_fred_series(series_id: str,
                      start_date: str = "1990-01-01",
                      end_date: str | None = None) -> pd.Series:

    if end_date is None:
        end_date = datetime.today().strftime("%Y-%m-%d")

    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": start_date,
        "observation_end": end_date,
    }

    # Send request
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()

    # Parse JSON payload
    data = response.json()["observations"]

    # Convert JSON → DataFrame → Series
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    s = df.set_index("date")["value"]
    s.name = series_id

    return s


In [3]:
# DOWNLOAD ALL MACRO SERIES
series_meta = {
    "core_cpi": "CPILFESL",       # Core CPI
    "fed_funds": "FEDFUNDS",      # Interest rates
    "unemployment": "UNRATE",     # Labor market
    "oil_price": "DCOILWTICO",    # Daily WTI oil price
    "m2": "M2SL",                 # Money supply
}

all_series = {}

for name, sid in series_meta.items():
    print(f"Fetching {name} ({sid})...")
    all_series[name] = fetch_fred_series(sid)

# Combine all series into a single DataFrame
macro_df = pd.concat(all_series.values(), axis=1)
macro_df.columns = list(series_meta.keys())

macro_df.tail()


Fetching core_cpi (CPILFESL)...
Fetching fed_funds (FEDFUNDS)...
Fetching unemployment (UNRATE)...
Fetching oil_price (DCOILWTICO)...
Fetching m2 (M2SL)...


,core_cpi,fed_funds,unemployment,oil_price,m2
date,,,,,
2026-04-07,NaN,NaN,NaN,114.58,NaN
2026-04-08,NaN,NaN,NaN,96.17,NaN
2026-04-09,NaN,NaN,NaN,99.62,NaN
2026-04-10,NaN,NaN,NaN,98.34,NaN
2026-04-13,NaN,NaN,NaN,100.72,NaN


In [4]:
# SAVING THE RAW DATA

os.makedirs("../data", exist_ok=True)

macro_df.to_parquet("../data/macro_raw.parquet")
print("Saved raw macro data to data/macro_raw.parquet")

Saved raw macro data to data/macro_raw.parquet
